# MPC Controller with Neural Network Dynamics Model (B747)

This example demonstrates the use of Model Predictive Control (MPC) with a neural network for approximating system dynamics.

## Main steps:
1. Create the B747 simulation environment
2. Create an MPC agent with a neural-network dynamics model
3. Collect data for dynamics model training
4. Train the dynamics model
5. Test the control system

Uses the linearized Boeing 747 longitudinal model.

## Import Required Libraries

Load all necessary modules for working with the MPC controller and simulation.

In [1]:
import numpy as np
import gymnasium as gym
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

from tensoraerospace.envs.b747 import LinearLongitudinalB747
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standart import unit_step
from tensoraerospace.agent.mpc import MPCAgent, MPCWeights, MPCConstraints

## Simulation Parameter Configuration

Set the time parameters and create the B747 simulation environment.

**State variables:**
- `u` -- forward velocity
- `w` -- vertical velocity
- `q` -- pitch rate
- `theta` -- pitch angle

**Control input:**
- `stab` -- stabilizer deflection

In [ ]:
# Time parameters
dt = 0.1
tp = generate_time_period(tn=20, dt=dt)
tps = convert_tp_to_sec_tp(tp, dt=dt)
number_time_steps = len(tp)

# Step at 5 seconds (tp is indices, not seconds; 5s / 0.1 dt = index 50)
step_index = int(5.0 / dt)  # = 50

# Reference signal -- 1-degree step
reference_signals_rad = np.reshape(
    unit_step(degree=1, tp=tp, time_step=step_index, output_rad=True),
    [1, -1]
)
# Env returns theta in degrees, so MPC x_ref needs degrees
reference_signals_deg = np.reshape(
    unit_step(degree=1, tp=tp, time_step=step_index, output_rad=False),
    [1, -1]
)

# Create B747 environment
env = gym.make(
    'LinearLongitudinalB747-v0',
    number_time_steps=number_time_steps,
    initial_state=[[0], [0], [0], [0]],
    reference_signal=reference_signals_rad,
    state_space=["u", "w", "q", "theta"],
    control_space=["stab"],
    output_space=["u", "w", "q", "theta"],
    tracking_states=["theta"],
    dt=dt,
)

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
print(f"State dim: {state_dim}, Action dim: {action_dim}")
print(f"Step starts at index {step_index} (t = {step_index * dt:.1f} s)")

## Creating the MPC Agent

Create an MPC agent with a neural-network dynamics model. The agent automatically:
- Creates an MLP network for dynamics approximation
- Configures the MPC solver with specified weights and constraints
- Manages the data buffer for training

**MPC parameters:**
- `horizon=20`: planning horizon
- `Q_diag`: state error weight (higher for theta)
- `R_diag`: control effort penalty
- `S_diag`: control rate penalty (smoothness)

In [ ]:
# MPC weights: high weight on theta (index 3), low on the rest
# R_diag and S_diag are increased to suppress parasitic oscillations before the step
weights = MPCWeights(
    Q_diag=np.array([0.0, 0.0, 0.5, 20.0], dtype=np.float32),
    R_diag=np.array([0.1], dtype=np.float32),
    S_diag=np.array([1.0], dtype=np.float32),
    terminal_weight=2.0,
)

# Control constraints
constraints = MPCConstraints(
    u_min=np.array([-25.0], dtype=np.float32),
    u_max=np.array([25.0], dtype=np.float32),
)

# Create MPC agent
agent = MPCAgent(
    env,
    horizon=20,
    weights=weights,
    constraints=constraints,
    tracking_type="tracking",
    iters=80,
    mpc_lr=0.03,
    hidden_layers=(256, 256),
    activation="relu",
    normalize=True,
    dynamics_lr=1e-3,
    memory_capacity=200_000,
    device="cpu",
    seed=42,
)

print("MPC agent created")

## Collecting Data for Dynamics Model Training

Collect transition data `(x, u, x_next)` by interacting with the environment.
Two collection rounds are used:
1. **Random actions** -- covering various operating points
2. **Structured signals** -- steps, sinusoids, chirps, etc. to capture dynamics at different frequencies and amplitudes

In [ ]:
# Round 1: random actions -- broad state-space coverage
agent.collect_data(
    num_episodes=100,
    exploration="random",
)
print(f"After random: {len(agent.memory)} transitions")

# Round 2: structured signals -- precise dynamics modeling
agent.collect_data(
    num_episodes=10000,
    exploration="signals",
    signal_kinds=[
        "random_steps",      # random steps of varying amplitude
        "unit_step",         # unit steps
        "multi_step",        # multiple steps per episode
        "ramp",              # linear ramp
        "sinusoid",          # sinusoids (primary dynamics)
        "multisine",         # sum of sinusoids (multiple frequencies)
        "chirp",             # frequency sweep
        "square_wave",       # square-wave pulses
        "triangular_wave",   # triangular waves
        "sawtooth",          # sawtooth waves
        "doublet",           # doublet pulses (sharp switches)
        "pulse",             # single pulses
        "gaussian_pulse",    # Gaussian pulses
        "damped_sinusoid",   # damped oscillations
    ],
    dt=dt,
)
print(f"After signals: {len(agent.memory)} transitions")

## Training the Dynamics Model

Train the neural network on collected data. The model learns to predict
the next system state `x_{t+1}` from the current state `x_t` and control input `u_t`.

In [ ]:
# Train dynamics model (more epochs for better accuracy)
result = agent.train_dynamics(
    epochs=300,
    batch_size=2048,
    loss="mse",
)

print(f"Final loss: {result['loss']:.8f}")

## Testing the MPC Controller

Create a new environment for testing and run the MPC controller.
At each step the agent:
1. Observes the current state
2. Forms a reference trajectory over the planning horizon
3. Optimizes the control signal via MPC
4. Applies the first element of the optimal sequence

In [ ]:
# Create test environment
test_env = gym.make(
    'LinearLongitudinalB747-v0',
    number_time_steps=number_time_steps,
    initial_state=[[0], [0], [0], [0]],
    reference_signal=reference_signals_rad,
    state_space=["u", "w", "q", "theta"],
    control_space=["stab"],
    output_space=["u", "w", "q", "theta"],
    tracking_states=["theta"],
    dt=dt,
)

# Run simulation
obs, info = test_env.reset()
agent.reset()

states_hist = [obs.flatten()]
controls_hist = []
horizon = agent.mpc.horizon

for step in tqdm(range(number_time_steps - 2)):
    # Current reference value (no lookahead)
    current_ref = reference_signals_deg[0, step]

    # x_ref: current reference repeated over the entire horizon
    # This prevents MPC from reacting in advance to future changes
    x_ref = np.zeros((horizon + 1, state_dim), dtype=np.float32)
    x_ref[:, 3] = current_ref  # theta = const over horizon

    action = agent.select_action(obs.flatten(), x_ref=x_ref)
    obs, reward, terminated, truncated, info = test_env.step(action)

    states_hist.append(obs.flatten())
    controls_hist.append(action)

    if terminated or truncated:
        break

states_hist = np.array(states_hist)
controls_hist = np.array(controls_hist)
print(f"Simulation complete: {len(controls_hist)} steps")

## Visualization of Results

Plot the transient response and control signal.

In [ ]:
n_steps = len(controls_hist)
time_array = np.arange(n_steps) * dt

# Theta angle from state history (index 3) -- already in degrees from env
theta_hist = states_hist[:n_steps, 3]
theta_ref = reference_signals_deg[0, :n_steps]

plt.figure(figsize=(15, 6))

plt.subplot(2, 1, 1)
plt.plot(time_array, theta_hist, label="MPC (theta)")
plt.plot(time_array, theta_ref, '--', label="Reference")
plt.ylabel("Theta (deg)")
plt.legend()
plt.title("MPC Controller Transient Response (B747)")
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(time_array, controls_hist)
plt.xlabel("Time (s)")
plt.ylabel("Control (deg)")
plt.title("Control Signal")
plt.grid(True)

plt.tight_layout()
plt.show()

In [9]:
from tensoraerospace.benchmark import ControlBenchmark

bench = ControlBenchmark()
res = bench.becnchmarking_one_step(theta_ref, theta_hist, 0.9, dt)
res

{'overshoot': 4.3616414070129395,
 'settling_time': 1.1,
 'damping_degree': 0.014909943,
 'static_error': 0.0031296610832214355,
 'rise_time': 0.8,
 'peak_time': 1.5,
 'maximum_deviation': 1.0024954,
 'iae': 7.466929513728246,
 'ise': 5.362765379504178,
 'itae': 5.9425986013840895,
 'oscillation_count': 2,
 'steady_state_value': 1.0,
 'performance_index': 5.394473873757895}

In [ ]:
print("Static error: ", res['static_error'])
print("Settling time: ", res['settling_time'], "s")
print("Damping degree: ", res['damping_degree'])
print("Overshoot: ", res['overshoot'])

## Detailed Transient Analysis

Use the built-in benchmark for control quality visualization.

In [11]:
bench.plot(theta_ref, theta_hist, 0.9, dt, time_array, figsize=(15, 5))

{'overshoot': 4.3616414070129395,
 'settling_time': 1.1,
 'damping_degree': 0.014909943,
 'static_error': 0.0031296610832214355,
 'rise_time': 0.8,
 'peak_time': 1.5,
 'maximum_deviation': 1.0024954,
 'iae': 7.466929513728246,
 'ise': 5.362765379504178,
 'itae': 5.9425986013840895,
 'oscillation_count': 2,
 'steady_state_value': 1.0,
 'performance_index': 5.394473873757895}

## Conclusion

This example demonstrated the complete development cycle of an MPC controller for the B747 aircraft:

1. **Data collection** -- gathered transitions from the environment for dynamics model training
2. **Neural network training** -- built a data-driven dynamics model
3. **MPC controller setup** -- configured the predictive control parameters
4. **System testing** -- verified control quality on test data

This approach can be adapted for other aircraft types and control tasks.